# Leisure Activity Metrics (Weekday)

Computes individual-level leisure metrics from weekday data only:
- `hill_q1`: Location diversity (effective number of unique H3 cells visited for LEISURE)
- `mean_leisure_duration`: Mean time spent at leisure locations (minutes)

Output: `dbs/data_p/commuter_leisure_metrics_weekday.csv`

In [1]:
%cd D:\netmob25

D:\netmob25


In [2]:
import pandas as pd
import numpy as np
from collections import Counter
from tqdm import tqdm
import h3.api.numpy_int as h3

## 1. Load and prepare trip data

In [3]:
# Load trip records
df = pd.read_parquet('dbs/data_p/stays_extraction_all.parquet')

# Parse timestamps and extract date
df['start_time'] = pd.to_datetime(df['start_time'])
df['end_time'] = pd.to_datetime(df['end_time'])
df['date'] = df['trip_id'].str.split('_').str[0]

# Sort for activity duration computation
df = df.sort_values(['ID', 'date', 'start_time']).reset_index(drop=True)

print(f'Total trips: {len(df)}')
print(f'Individuals: {df["ID"].nunique()}')

Total trips: 69706
Individuals: 3318


In [4]:
# Compute activity duration at each location
# (time between arriving and departing = start of next trip - end of previous trip)
prev_end = df.groupby(['ID', 'date'])['end_time'].shift(1)
df['activity_duration_min'] = (df['start_time'] - prev_end).dt.total_seconds() / 60.0
df['activity_duration_min'] = df['activity_duration_min'].clip(lower=0)

In [5]:
# Filter to WEEKDAYS only
df_wk = df[~df['dow'].isin(['saturday', 'sunday'])].copy()

print(f'Weekday trips: {len(df_wk)} ({len(df_wk)/len(df)*100:.1f}%)')
print(f'Individuals with weekday data: {df_wk["ID"].nunique()}')

Weekday trips: 54868 (78.7%)
Individuals with weekday data: 3314


In [6]:
# Separate leisure-related subsets
# Trips TO leisure (for counting trips and computing location diversity)
df_to_leisure = df_wk[df_wk['purpose_d'] == 'LEISURE'].copy()

# Trips FROM leisure (for computing duration spent at leisure)
df_from_leisure = df_wk[df_wk['purpose_o'] == 'LEISURE'].copy()

print(f'Trips to LEISURE: {len(df_to_leisure)}')
print(f'Trips from LEISURE: {len(df_from_leisure)}')
print(f'Individuals with weekday leisure: {df_to_leisure["ID"].nunique()}')

Trips to LEISURE: 4579
Trips from LEISURE: 4499
Individuals with weekday leisure: 2015


## 2. Define metrics computation

In [7]:
def compute_leisure_metrics(id_, df_all, df_to_leisure, df_from_leisure):
    """
    Compute leisure metrics for one individual.
    
    Parameters:
    - id_: Individual ID
    - df_all: All weekday trips for this individual
    - df_to_leisure: Trips with destination=LEISURE
    - df_from_leisure: Trips with origin=LEISURE (for duration)
    
    Returns dict with:
    - n_weekdays: Number of observation days
    - n_leisure_trips: Count of trips to leisure
    - n_leisure_days: Days with at least one leisure trip
    - hill_q1: Location diversity (effective number of unique H3-10 cells)
    - mean_leisure_duration: Mean time spent at leisure locations (min)
    - total_leisure_duration: Total time spent at leisure (min)
    """
    grp_all = df_all[df_all['ID'] == id_]
    grp_to = df_to_leisure[df_to_leisure['ID'] == id_]
    grp_from = df_from_leisure[df_from_leisure['ID'] == id_]
    
    n_weekdays = grp_all['date'].nunique()
    n_trips = len(grp_to)
    n_leisure_days = grp_to['date'].nunique() if n_trips > 0 else 0
    
    # --- Hill q1: from destination coordinates ---
    if n_trips > 0:
        # Get valid destination coordinates
        coords = [(row['end_lat'], row['end_lon']) 
                  for _, row in grp_to.iterrows()
                  if pd.notna(row['end_lat']) and pd.notna(row['end_lon'])]
        
        if len(coords) > 0:
            # Convert to H3 resolution 10
            h3_cells = [h3.latlng_to_cell(lat, lon, res=10) for lat, lon in coords]
            
            # Count frequencies
            cell_counts = Counter(h3_cells)
            counts = np.array(list(cell_counts.values()), dtype=float)
            K = len(counts)  # unique locations
            
            if K > 1:
                p = counts / counts.sum()
                hill_q1 = np.exp(-np.sum(p * np.log(p)))
            else:
                hill_q1 = 1.0
        else:
            hill_q1 = np.nan
    else:
        hill_q1 = np.nan
    
    # --- Duration: from trips departing leisure ---
    if len(grp_from) > 0:
        durations = grp_from['activity_duration_min'].dropna()
        if len(durations) > 0:
            total_duration = durations.sum()
            mean_duration = durations.mean()
        else:
            total_duration = 0
            mean_duration = np.nan
    else:
        total_duration = 0
        mean_duration = np.nan
    
    return {
        'ID': id_,
        'n_weekdays': n_weekdays,
        'n_leisure_trips': n_trips,
        'n_leisure_days': n_leisure_days,
        'hill_q1': hill_q1,
        'total_leisure_duration': total_duration,
        'mean_leisure_duration': mean_duration,
    }

## 3. Compute metrics for all individuals

In [8]:
# Get all individual IDs with weekday data
all_ids = df_wk['ID'].unique()

# Compute metrics
metrics_list = []
for id_ in tqdm(all_ids, desc='Computing leisure metrics'):
    metrics_list.append(compute_leisure_metrics(id_, df_wk, df_to_leisure, df_from_leisure))

df_metrics = pd.DataFrame(metrics_list)
print(f'\nComputed metrics for {len(df_metrics)} individuals')

Computing leisure metrics: 100%|██████████| 3314/3314 [00:13<00:00, 240.40it/s]


Computed metrics for 3314 individuals


In [9]:
# Summary statistics
print('=== Leisure Metrics Summary ===')
print(df_metrics.describe().round(2))

=== Leisure Metrics Summary ===
       n_weekdays  n_leisure_trips  n_leisure_days  hill_q1  \
count     3314.00          3314.00         3314.00  2015.00   
mean         4.36             1.38            1.13     2.20   
std          0.88             1.70            1.22     1.53   
min          1.00             0.00            0.00     1.00   
25%          4.00             0.00            0.00     1.00   
50%          5.00             1.00            1.00     2.00   
75%          5.00             2.00            2.00     3.00   
max          6.00            13.00            5.00    13.00   

       total_leisure_duration  mean_leisure_duration  
count                 3314.00                1980.00  
mean                   152.81                 121.27  
std                    228.21                 102.80  
min                      0.00                   0.00  
25%                      0.00                  63.68  
50%                     68.00                 101.81  
75%            

In [10]:
# Check correlations
print('=== Correlations ===')
cols = ['n_leisure_trips', 'n_leisure_days', 'hill_q1', 'mean_leisure_duration']
print(df_metrics[cols].corr().round(3))

=== Correlations ===
                       n_leisure_trips  n_leisure_days  hill_q1  \
n_leisure_trips                  1.000           0.925    0.972   
n_leisure_days                   0.925           1.000    0.854   
hill_q1                          0.972           0.854    1.000   
mean_leisure_duration           -0.182          -0.173   -0.104   

                       mean_leisure_duration  
n_leisure_trips                       -0.182  
n_leisure_days                        -0.173  
hill_q1                               -0.104  
mean_leisure_duration                  1.000  


In [11]:
# Data availability
print('=== Data Availability ===')
print(f'Total individuals: {len(df_metrics)}')
print(f'With any leisure (hill_q1 valid): {df_metrics["hill_q1"].notna().sum()} ({df_metrics["hill_q1"].notna().mean()*100:.1f}%)')
print(f'With duration data: {df_metrics["mean_leisure_duration"].notna().sum()} ({df_metrics["mean_leisure_duration"].notna().mean()*100:.1f}%)')

=== Data Availability ===
Total individuals: 3314
With any leisure (hill_q1 valid): 2015 (60.8%)
With duration data: 1980 (59.7%)


## 4. Filter to commuters and save

In [12]:
# Load commuter IDs from existing model features
df_comm = pd.read_csv('dbs/data_p/commuter_model_features_r.csv')
commuter_ids = df_comm['ID'].unique()

# Filter metrics to commuters
df_metrics_comm = df_metrics[df_metrics['ID'].isin(commuter_ids)].copy()

print(f'Commuters with weekday data: {len(df_metrics_comm)}')
print(f'With hill_q1: {df_metrics_comm["hill_q1"].notna().sum()}')
print(f'With mean_leisure_duration: {df_metrics_comm["mean_leisure_duration"].notna().sum()}')

Commuters with weekday data: 2444
With hill_q1: 1454
With mean_leisure_duration: 1429


In [13]:
# Save output
output_path = 'dbs/data_p/commuter_leisure_metrics_weekday.csv'
df_metrics_comm.to_csv(output_path, index=False)
print(f'Saved to: {output_path}')

Saved to: dbs/data_p/commuter_leisure_metrics_weekday.csv


In [14]:
# Preview output
print('=== Output Preview ===')
print(df_metrics_comm.head(10))

=== Output Preview ===
         ID  n_weekdays  n_leisure_trips  n_leisure_days  hill_q1  \
0   10_2978           3                0               0      NaN   
1   10_2980           5                1               1      1.0   
2   10_2981           4                0               0      NaN   
3   10_2982           5                2               2      2.0   
4   10_2984           6                2               2      2.0   
6   10_2988           4                1               1      1.0   
7   10_2989           5                0               0      NaN   
10  10_2992           4                0               0      NaN   
11  10_2993           5                0               0      NaN   
12  10_2994           5                0               0      NaN   

    total_leisure_duration  mean_leisure_duration  
0                 0.000000                    NaN  
1               216.016667             216.016667  
2                 0.000000                    NaN  
3        

## 5. Verify key findings

Confirm that hill_q1 and mean_leisure_duration capture different dimensions.

In [15]:
# Correlation between outcomes
valid = df_metrics_comm.dropna(subset=['hill_q1', 'mean_leisure_duration'])
r = valid['hill_q1'].corr(valid['mean_leisure_duration'])
print(f'Correlation(hill_q1, mean_leisure_duration) = {r:.3f}')
print(f'Shared variance = {r**2*100:.1f}%')
print(f'\nThese metrics are nearly orthogonal (r={r:.2f}), capturing different dimensions:')
print('- hill_q1: breadth/diversity of leisure engagement')
print('- mean_leisure_duration: depth/intensity per visit')

Correlation(hill_q1, mean_leisure_duration) = -0.111
Shared variance = 1.2%

These metrics are nearly orthogonal (r=-0.11), capturing different dimensions:
- hill_q1: breadth/diversity of leisure engagement
- mean_leisure_duration: depth/intensity per visit
